# Assignment 06 — Weather Analysis for Agricultural Fields

**Objective:** Analyze NASA POWER daily weather data for 20 Ohio Maumee watershed fields (2020–2023), calculate Growing Degree Days (GDD), characterize seasonal climate patterns, and explore relationships between weather variables, crop type, and soil properties.

**Key tasks:**
1. Load and inspect NASA POWER weather data for the study fields
2. Explore temperature, precipitation, solar radiation, and humidity distributions
3. Calculate Growing Degree Days (GDD) for corn and soybeans
4. Analyze seasonal and year-over-year climate trends
5. Compare growing-season weather across crop types
6. Visualize temperature ranges, precipitation patterns, and GDD accumulation

**Data sources:**
- `data/weather/ohio_maumee_20_2020_2023.csv` — Daily weather data (NASA POWER, 2020–2023)
- `data/fields/ohio_maumee_20.geojson` — 20 field boundaries (EPSG:4326)
- `data/cdl/ohio_maumee_20_cdl.csv` — Crop type classifications (2020–2023)
- `data/soil/ohio_maumee_20_soil.csv` — SSURGO soil data

**Weather variables (NASA POWER):**

| Parameter | Description | Units |
|-----------|-------------|-------|
| `T2M` | Daily mean temperature at 2 m | °C |
| `T2M_MAX` | Daily maximum temperature at 2 m | °C |
| `T2M_MIN` | Daily minimum temperature at 2 m | °C |
| `PRECTOTCORR` | Precipitation (bias-corrected) | mm/day |
| `ALLSKY_SFC_SW_DWN` | Solar radiation (shortwave downward) | MJ/m²/day |
| `RH2M` | Relative humidity at 2 m | % |

## 1. Setup and imports

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os

# Consistent plot style
sns.set_theme(style='whitegrid', font_scale=1.2)

# Output directory
os.makedirs('output', exist_ok=True)

print('Setup complete.')

## 2. Load datasets

In [ ]:
# Weather data
weather = pd.read_csv('../data/weather/ohio_maumee_20_2020_2023.csv')

# Ensure the date column is in datetime format
weather['date'] = pd.to_datetime(weather['date'], format='%Y-%m-%d')
assert pd.api.types.is_datetime64_any_dtype(weather['date']), 'date column is not datetime!'
print(f'Weather: {weather.shape[0]:,} records, {weather["field_id"].nunique()} fields')
print(f'Date range: {weather["date"].min().date()} to {weather["date"].max().date()}')
print(f'Date dtype: {weather["date"].dtype}')

# Field boundaries (optional — file may not exist yet)
fields_path = '../data/fields/ohio_maumee_20.geojson'
if os.path.exists(fields_path):
    fields = gpd.read_file(fields_path)
    print(f'Fields: {fields.shape[0]} fields, CRS: {fields.crs}')
else:
    fields = None
    print(f'Fields file not found at {fields_path} — skipping.')

# Crop data — fix quoted/spaced field_id values in the CDL CSV
cdl = pd.read_csv('../data/cdl/ohio_maumee_20_cdl.csv')
cdl['field_id'] = (
    cdl['field_id']
    .astype(str)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip('"')
    .astype(int)
)
print(f'CDL: {cdl.shape[0]} records, {cdl["field_id"].nunique()} unique fields')

# Soil data (optional — file may not exist yet)
soil_path = '../data/soil/ohio_maumee_20_soil.csv'
if os.path.exists(soil_path):
    soil = pd.read_csv(soil_path)
    print(f'Soil: {soil.shape[0]} records')
else:
    soil = None
    print(f'Soil file not found at {soil_path} — skipping.')

In [ ]:
# Inspect weather data
weather.head()

In [ ]:
weather.info()

In [ ]:
weather.describe()

## 3. Data quality checks

In [ ]:
# Check for missing values
missing = weather.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values.')

# Check for NASA POWER sentinel missing value (-999)
sentinel_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'ALLSKY_SFC_SW_DWN', 'RH2M']
for col in sentinel_cols:
    n_bad = (weather[col] == -999.0).sum()
    if n_bad > 0:
        print(f'  {col}: {n_bad} sentinel (-999) values')

# Replace sentinel values with NaN
weather[sentinel_cols] = weather[sentinel_cols].replace(-999.0, np.nan)
print(f'\nRecords after cleaning: {weather.shape[0]:,}')

In [ ]:
# Verify date coverage per field
coverage = weather.groupby('field_id')['date'].agg(['min', 'max', 'count'])
coverage.columns = ['first_date', 'last_date', 'n_days']
print(f'Days per field: min={coverage["n_days"].min()}, max={coverage["n_days"].max()}')
coverage.head()

## 3a. Filter to most recent growing season

The growing season for Ohio corn and soybeans is typically **May through September**.
We identify the most recent year in the data and filter to that growing season so all
downstream analysis reflects current-season conditions.

In [ ]:
# Identify the most recent year in the dataset
latest_year = weather['date'].dt.year.max()
print(f'Most recent year in data: {latest_year}')

# Define growing-season months (May–September)
GROWING_MONTHS = [5, 6, 7, 8, 9]

# Filter to the most recent growing season
weather_gs = weather[
    (weather['date'].dt.year == latest_year) &
    (weather['date'].dt.month.isin(GROWING_MONTHS))
].copy()

# Add convenience columns
weather_gs['month'] = weather_gs['date'].dt.month
weather_gs['year'] = weather_gs['date'].dt.year

print(f'Filtered to {latest_year} growing season (May–Sep):')
print(f'  Records: {weather_gs.shape[0]:,}')
print(f'  Fields:  {weather_gs["field_id"].nunique()}')
print(f'  Date range: {weather_gs["date"].min().date()} to {weather_gs["date"].max().date()}')
weather_gs.head()

## 4. Temperature analysis

In [ ]:
# Distribution of daily temperature during the most recent growing season
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, label in zip(axes,
                          ['T2M_MIN', 'T2M', 'T2M_MAX'],
                          ['Daily Min', 'Daily Mean', 'Daily Max']):
    ax.hist(weather_gs[col].dropna(), bins=40, edgecolor='white', alpha=0.8)
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('Frequency')
    ax.set_title(label)
    ax.axvline(weather_gs[col].mean(), color='red', linestyle='--',
               label=f'Mean: {weather_gs[col].mean():.1f}°C')
    ax.legend()

fig.suptitle(f'Temperature Distributions — All Fields ({latest_year} Growing Season, May–Sep)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('output/06_temperature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_temperature_distributions.png')

In [ ]:
# Monthly mean temperature across all fields
weather['month'] = weather['date'].dt.month
weather['year'] = weather['date'].dt.year

monthly_temp = weather.groupby(['year', 'month'])['T2M'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for yr in sorted(monthly_temp['year'].unique()):
    subset = monthly_temp[monthly_temp['year'] == yr]
    ax.plot(subset['month'], subset['T2M'], marker='o', label=str(yr))

ax.set_xlabel('Month')
ax.set_ylabel('Mean Temperature (°C)')
ax.set_title('Monthly Mean Temperature by Year')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.legend(title='Year')
plt.tight_layout()
plt.savefig('output/06_monthly_temperature.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_monthly_temperature.png')

In [ ]:
# Temperature range (min/max envelope) for a single representative field — most recent growing season
sample_field = weather_gs['field_id'].unique()[0]
yr_data = weather_gs[weather_gs['field_id'] == sample_field].sort_values('date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(yr_data['date'], yr_data['T2M_MIN'], yr_data['T2M_MAX'],
                alpha=0.3, label='Min / Max range')
ax.plot(yr_data['date'], yr_data['T2M'], linewidth=0.8, label='Daily mean')
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.set_ylabel('Temperature (°C)')
ax.set_title(f'Daily Temperature Range — Field {sample_field} ({latest_year} Growing Season)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.savefig('output/06_temperature_range_growing_season.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_temperature_range_growing_season.png')

## 4b. Time-series: Daily temperature and precipitation

Combined time-series showing daily high/low temperature and precipitation for all field locations
during the most recent growing season.

In [ ]:
# Aggregate daily weather across all fields (mean values)
daily_weather = weather_gs.groupby('date').agg({
    'T2M_MIN': 'mean',
    'T2M_MAX': 'mean',
    'T2M': 'mean',
    'PRECTOTCORR': 'sum'
}).reset_index()

daily_weather = daily_weather.sort_values('date')

print(f'Daily aggregated weather: {daily_weather.shape[0]} days')
daily_weather.head()

In [ ]:
# Time-series: Daily temperature range + precipitation (dual axis)
fig, ax1 = plt.subplots(figsize=(16, 6))

# Temperature: high-low range as shaded area
ax1.fill_between(daily_weather['date'], 
                 daily_weather['T2M_MIN'], 
                 daily_weather['T2M_MAX'],
                 alpha=0.3, color='coral', label='Temperature Range (Min–Max)')
ax1.plot(daily_weather['date'], daily_weather['T2M'], 
         color='red', linewidth=1.2, label='Daily Mean Temp')
ax1.set_xlabel('Date')
ax1.set_ylabel('Temperature (°C)', color='red')
ax1.tick_params(axis='y', labelcolor='red')
ax1.set_ylim(-5, 40)

# Precipitation: bar chart on secondary axis
ax2 = ax1.twinx()
ax2.bar(daily_weather['date'], daily_weather['PRECTOTCORR'], 
        width=0.8, color='steelblue', alpha=0.6, label='Daily Precipitation')
ax2.set_ylabel('Precipitation (mm)', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')
ax2.set_ylim(0, daily_weather['PRECTOTCORR'].max() * 3)

# Formatting
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title(f'Daily Temperature Range & Precipitation — {latest_year} Growing Season (All Fields)',
          fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('output/06_daily_temp_precip_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_daily_temp_precip_timeseries.png')

In [ ]:
# Time-series faceted by field: daily temp range + precip for each field
n_fields = weather_gs['field_id'].nunique()
fig, axes = plt.subplots(n_fields, 1, figsize=(14, 3 * n_fields), sharex=True)

for ax, fid in zip(axes, sorted(weather_gs['field_id'].unique())):
    fdata = weather_gs[weather_gs['field_id'] == fid].sort_values('date')
    
    # Temperature range
    ax.fill_between(fdata['date'], fdata['T2M_MIN'], fdata['T2M_MAX'],
                    alpha=0.3, color='coral')
    ax.plot(fdata['date'], fdata['T2M'], color='red', linewidth=0.8)
    
    # Precipitation bars
    ax2 = ax.twinx()
    ax2.bar(fdata['date'], fdata['PRECTOTCORR'], width=0.8, 
            color='steelblue', alpha=0.5)
    ax2.set_ylim(0, fdata['PRECTOTCORR'].max() * 3 if fdata['PRECTOTCORR'].max() > 0 else 10)
    
    ax.set_ylabel('Temp (°C)', fontsize=9)
    ax2.set_ylabel('Precip (mm)', fontsize=9)
    ax.set_title(f'Field {fid}', fontsize=10, loc='left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

axes[-1].set_xlabel('Date')
plt.suptitle(f'Daily Temperature & Precipitation by Field — {latest_year} Growing Season',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('output/06_daily_temp_precip_by_field.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_daily_temp_precip_by_field.png')

## 4c. Rolling averages — smoothed seasonal trends

Calculate 7-day and 30-day rolling averages to smooth daily fluctuations and highlight
broader seasonal patterns in temperature and precipitation.

In [ ]:
# Aggregate daily weather across all fields
daily_weather = weather_gs.groupby('date').agg({
    'T2M_MIN': 'mean',
    'T2M_MAX': 'mean',
    'T2M': 'mean',
    'PRECTOTCORR': 'sum'
}).reset_index().sort_values('date')

# Calculate rolling averages (centered)
daily_weather['T2M_7d'] = daily_weather['T2M'].rolling(window=7, center=True).mean()
daily_weather['T2M_30d'] = daily_weather['T2M'].rolling(window=30, center=True).mean()
daily_weather['precip_7d'] = daily_weather['PRECTOTCORR'].rolling(window=7, center=True).mean()
daily_weather['precip_30d'] = daily_weather['PRECTOTCORR'].rolling(window=30, center=True).mean()

print(f'Rolling averages calculated: 7-day and 30-day windows')
daily_weather[['date', 'T2M', 'T2M_7d', 'T2M_30d', 'precip_7d', 'precip_30d']].head(35)

In [ ]:
# Visualization: Rolling averages for temperature and precipitation
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Top: Temperature with rolling averages
ax1 = axes[0]
ax1.plot(daily_weather['date'], daily_weather['T2M'], alpha=0.3, color='gray', linewidth=0.8, label='Daily Mean')
ax1.plot(daily_weather['date'], daily_weather['T2M_7d'], color='orange', linewidth=2.5, label='7-Day Rolling Avg')
ax1.plot(daily_weather['date'], daily_weather['T2M_30d'], color='darkred', linewidth=3, label='30-Day Rolling Avg')
ax1.set_ylabel('Temperature (°C)', fontsize=12)
ax1.set_title(f'Daily Mean Temperature with Rolling Averages — {latest_year} Growing Season', fontsize=13)
ax1.legend(loc='upper left')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax1.set_ylim(5, 35)

# Bottom: Precipitation with rolling averages
ax2 = axes[1]
ax2.bar(daily_weather['date'], daily_weather['PRECTOTCORR'], alpha=0.3, color='steelblue', width=0.8, label='Daily Precip')
ax2.plot(daily_weather['date'], daily_weather['precip_7d'], color='orange', linewidth=2.5, label='7-Day Rolling Avg')
ax2.plot(daily_weather['date'], daily_weather['precip_30d'], color='darkblue', linewidth=3, label='30-Day Rolling Avg')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Precipitation (mm)', fontsize=12)
ax2.set_title(f'Daily Precipitation with Rolling Averages — {latest_year} Growing Season', fontsize=13)
ax2.legend(loc='upper left')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('output/06_rolling_averages_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_rolling_averages_combined.png')

In [ ]:
# Visualization: Smoothed trends only (cleaner view)
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Temperature trends only
ax1 = axes[0]
ax1.fill_between(daily_weather['date'], 
                daily_weather['T2M_MIN'].rolling(7, center=True).mean(), 
                daily_weather['T2M_MAX'].rolling(7, center=True).mean(), 
                alpha=0.2, color='coral', label='7d Temp Range')
ax1.plot(daily_weather['date'], daily_weather['T2M_7d'], color='red', linewidth=2.5, label='7-Day Avg')
ax1.plot(daily_weather['date'], daily_weather['T2M_30d'], color='darkred', linewidth=3, linestyle='--', label='30-Day Avg')
ax1.set_ylabel('Temperature (°C)', fontsize=12)
ax1.set_title(f'Smoothed Temperature Trends — {latest_year} Growing Season', fontsize=13)
ax1.legend(loc='upper left')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.set_ylim(10, 32)
ax1.grid(True, alpha=0.3)

# Precipitation trends only
ax2 = axes[1]
ax2.plot(daily_weather['date'], daily_weather['precip_7d'], color='steelblue', linewidth=2.5, label='7-Day Rolling Avg')
ax2.plot(daily_weather['date'], daily_weather['precip_30d'], color='darkblue', linewidth=3, linestyle='--', label='30-Day Rolling Avg')
ax2.fill_between(daily_weather['date'], 0, daily_weather['precip_7d'], alpha=0.2, color='steelblue')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Precipitation (mm/day)', fontsize=12)
ax2.set_title(f'Smoothed Precipitation Trends — {latest_year} Growing Season', fontsize=13)
ax2.legend(loc='upper left')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/06_rolling_averages_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_rolling_averages_trends.png')

## 4d. Anomaly detection — comparing 2023 to historical averages

Compare the most recent year's (2023) growing-season weather against the historical average
from prior years (2020–2022) to identify anomalies such as drought, late frost, or heat waves.

In [ ]:
# Compare current year to historical averages
prior_years = [y for y in weather['year'].unique() if y < latest_year]

# Historical monthly averages (prior years)
weather_hist = weather[(weather['year'].isin(prior_years)) & (weather['month'].isin(GROWING_MONTHS))]
hist_monthly = weather_hist.groupby('month').agg({
    'T2M': 'mean',
    'T2M_MIN': 'mean',
    'T2M_MAX': 'mean',
    'PRECTOTCORR': 'sum'
}).reset_index()
hist_monthly.columns = ['month', 'hist_T2M', 'hist_T2M_MIN', 'hist_T2M_MAX', 'hist_precip']

# Current year monthly data
weather_current = weather[(weather['year'] == latest_year) & (weather['month'].isin(GROWING_MONTHS))]
current_monthly = weather_current.groupby('month').agg({
    'T2M': 'mean',
    'T2M_MIN': 'mean',
    'T2M_MAX': 'mean',
    'PRECTOTCORR': 'sum'
}).reset_index()
current_monthly.columns = ['month', 'curr_T2M', 'curr_T2M_MIN', 'curr_T2M_MAX', 'curr_precip']

# Merge and calculate anomalies
comparison = hist_monthly.merge(current_monthly, on='month')
comparison['temp_anomaly'] = comparison['curr_T2M'] - comparison['hist_T2M']
comparison['precip_anomaly'] = comparison['curr_precip'] - comparison['hist_precip']
comparison['precip_pct_of_norm'] = (comparison['curr_precip'] / comparison['hist_precip']) * 100
comparison['month_name'] = comparison['month'].apply(lambda x: ['May','Jun','Jul','Aug','Sep'][x-5])

print(f'=== {latest_year} vs Historical Average ({prior_years}) ===')
print('\nTemperature Anomaly (°C):')
print(comparison[['month_name', 'hist_T2M', 'curr_T2M', 'temp_anomaly']].to_string(index=False))

print('\nPrecipitation Anomaly (mm):')
print(comparison[['month_name', 'hist_precip', 'curr_precip', 'precip_anomaly', 'precip_pct_of_norm']].to_string(index=False))

In [ ]:
# Anomaly detection summary
print(f'\n=== Anomaly Detection Results ===')

# Check for late-season frost (May min temp)
may_min = comparison[comparison['month'] == 5]['curr_T2M_MIN'].values[0]
print(f'May minimum temperature: {may_min:.1f}°C')
if may_min < 0:
    print('  ⚠️ LATE-SEASON FROST DETECTED!')
else:
    print('  ✓ No late-season frost')

# Check for prolonged dry spell
dry_months = comparison[comparison['precip_pct_of_norm'] < 50]
if len(dry_months) > 0:
    print(f'\n⚠️ PROLONGED DRY SPELL: {len(dry_months)} month(s) with <50% normal precipitation:')
    for _, row in dry_months.iterrows():
        print(f'   - {row["month_name"]}: {row["precip_pct_of_norm"]:.0f}% of normal ({row["curr_precip"]:.1f}mm vs {row["hist_precip"]:.1f}mm)')
else:
    print('\n✓ No prolonged dry spells detected')

# Temperature extremes
hottest = comparison.loc[comparison['temp_anomaly'].idxmax()]
coldest = comparison.loc[comparison['temp_anomaly'].idxmin()]
print(f'\nTemperature extremes:')
print(f'   Warmest vs normal: {hottest["month_name"]} ({hottest["temp_anomaly"]:+.1f}°C)')
print(f'   Coolest vs normal: {coldest["month_name"]} ({coldest["temp_anomaly"]:+.1f}°C)')

In [ ]:
# Visualization: Precipitation deficit bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Precipitation - actual vs normal
ax1 = axes[0]
x = np.arange(len(comparison))
width = 0.35

bars1 = ax1.bar(x - width/2, comparison['hist_precip'], width, label='Historical Avg', color='steelblue', alpha=0.7)
bars2 = ax1.bar(x + width/2, comparison['curr_precip'], width, label=f'{latest_year}', color='coral', alpha=0.9)

ax1.set_xlabel('Month')
ax1.set_ylabel('Total Precipitation (mm)')
ax1.set_title('Monthly Precipitation: 2023 vs Historical Average')
ax1.set_xticks(x)
ax1.set_xticklabels(comparison['month_name'])
ax1.legend()

# Right: Precipitation anomaly (deficit)
ax2 = axes[1]
colors = ['darkred' if x < 0 else 'steelblue' for x in comparison['precip_anomaly']]
bars = ax2.bar(comparison['month_name'], comparison['precip_anomaly'], color=colors, edgecolor='black', linewidth=1)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.set_xlabel('Month')
ax2.set_ylabel('Precipitation Anomaly (mm)')
ax2.set_title('2023 Precipitation Anomaly vs Historical Average\n(Deficit shown in red)')

plt.tight_layout()
plt.savefig('output/06_precipitation_deficit_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_precipitation_deficit_bars.png')

In [ ]:
# Visualization: Percentage of normal precipitation (drought indicator)
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#d73027' if x < 25 else '#fc8d59' if x < 50 else '#fee090' if x < 75 else '#91bfdb' if x < 100 else '#4575b4' 
          for x in comparison['precip_pct_of_norm']]
          
bars = ax.bar(comparison['month_name'], comparison['precip_pct_of_norm'], color=colors, edgecolor='black', linewidth=1.5)

ax.axhline(y=100, color='green', linestyle='--', linewidth=2, label='Normal (100%)')
ax.axhline(y=50, color='orange', linestyle='--', linewidth=1.5, label='Drought threshold (50%)')

for bar, pct in zip(bars, comparison['precip_pct_of_norm']):
    height = bar.get_height()
    ax.annotate(f'{pct:.0f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 5), textcoords='offset points', ha='center', va='bottom', 
                fontsize=12, fontweight='bold', color='black')

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('% of Normal Precipitation', fontsize=12)
ax.set_title(f'{latest_year} Growing Season Precipitation as % of Historical Normal\n⚠️ SEVERE DROUGHT: All months below 50% of normal', fontsize=13)
ax.set_ylim(0, 120)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('output/06_precipitation_pct_of_normal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_precipitation_pct_of_normal.png')

In [ ]:
# Visualization: Temperature anomaly
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#2166ac' if x < 0 else '#b2182b' for x in comparison['temp_anomaly']]
bars = ax.bar(comparison['month_name'], comparison['temp_anomaly'], color=colors, edgecolor='black', linewidth=1)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)

for bar, val in zip(bars, comparison['temp_anomaly']):
    height = bar.get_height()
    va = 'bottom' if height >= 0 else 'top'
    offset = 3 if height >= 0 else -3
    ax.annotate(f'{val:+.1f}°C', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, offset), textcoords='offset points', ha='center', va=va, fontsize=11, fontweight='bold')

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Temperature Anomaly (°C)', fontsize=12)
ax.set_title(f'{latest_year} Temperature Anomaly vs Historical Average ({prior_years})\nCooler than normal summer months', fontsize=13)

plt.tight_layout()
plt.savefig('output/06_temperature_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_temperature_anomaly.png')

## 4e. Dashboard assets — optimized weather visualizations

These plots are designed for dashboard integration, focusing on the most critical weather factors for row crops:
- **Cumulative precipitation** — tracking water availability through the season
- **Cumulative GDD (Growing Degree Days)** — tracking heat unit accumulation for crop development

Saved to: `output/dashboard_assets/`

In [ ]:
# Prepare data for dashboard plots
os.makedirs('output/dashboard_assets', exist_ok=True)

# Aggregate by date across all fields
daily_agg = weather_gs_gdd.groupby('date').agg({
    'PRECTOTCORR': 'sum',
    'gdd_cumulative': 'mean'
}).reset_index().sort_values('date')

daily_agg['precip_cumulative'] = daily_agg['PRECTOTCORR'].cumsum()
daily_agg = daily_agg.reset_index(drop=True)

print(f'Dashboard data prepared: {daily_agg.shape[0]} days')

In [ ]:
# Dashboard Plot 1: Cumulative Precipitation (clean, minimalist)
fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(daily_agg['date'], 0, daily_agg['precip_cumulative'], 
                alpha=0.3, color='#2196F3')
ax.plot(daily_agg['date'], daily_agg['precip_cumulative'], 
        color='#1565C0', linewidth=3, marker='o', markersize=3)

ax.axhline(y=daily_agg['precip_cumulative'].max(), color='gray', 
           linestyle='--', alpha=0.7, linewidth=1)
ax.text(daily_agg['date'].iloc[-1], daily_agg['precip_cumulative'].max() + 100,
        f"Total: {daily_agg['precip_cumulative'].max():.0f} mm", 
        fontsize=11, fontweight='bold', color='#1565C0')

ax.set_facecolor('#f8f9fa')
ax.grid(True, alpha=0.3, linestyle='-')
ax.set_ylabel('Cumulative Precipitation (mm)', fontsize=12, fontweight='bold', color='#1565C0')
ax.set_title(f'{latest_year} Growing Season Cumulative Precipitation\n(Aggregated across 20 fields)', 
             fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(fontsize=11)
plt.yticks(fontsize=10)
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig('output/dashboard_assets/cumulative_precip_weather_trends.png', 
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved output/dashboard_assets/cumulative_precip_weather_trends.png')

In [ ]:
# Dashboard Plot 2: Dual-axis (Cumulative Precip + GDD)
fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = '#1565C0'
ax1.fill_between(daily_agg['date'], 0, daily_agg['precip_cumulative'], 
                alpha=0.25, color=color1)
line1 = ax1.plot(daily_agg['date'], daily_agg['precip_cumulative'], 
                 color=color1, linewidth=2.5, label='Cumulative Precipitation')
ax1.set_xlabel('Date', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cumulative Precipitation (mm)', fontsize=12, fontweight='bold', color=color1)
ax1.tick_params(axis='y', labelcolor=color1, labelsize=10)
ax1.set_ylim(0, daily_agg['precip_cumulative'].max() * 1.15)

ax2 = ax1.twinx()
color2 = '#EF6C00'
line2 = ax2.plot(daily_agg['date'], daily_agg['gdd_cumulative'], 
                 color=color2, linewidth=2.5, linestyle='--', label='Cumulative GDD')
ax2.set_ylabel('Cumulative GDD (base 10°C)', fontsize=12, fontweight='bold', color=color2)
ax2.tick_params(axis='y', labelcolor=color2, labelsize=10)
ax2.set_ylim(0, daily_agg['gdd_cumulative'].max() * 1.15)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=10, framealpha=0.9)

ax1.set_facecolor('#f8f9fa')
ax1.grid(True, alpha=0.3, linestyle='-')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.set_title(f'{latest_year} Growing Season: Precipitation & Heat Units\n(Cumulative metrics across 20 fields)', 
             fontsize=14, fontweight='bold', pad=15)

for spine in ax1.spines.values():
    spine.set_visible(False)
for spine in ax2.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig('output/dashboard_assets/precip_gdd_weather_trends.png', 
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved output/dashboard_assets/precip_gdd_weather_trends.png')

## 5. Precipitation analysis

In [ ]:
# Monthly precipitation totals for the most recent growing season
gs_monthly_precip = (weather_gs
    .groupby(['field_id', 'month'])['PRECTOTCORR']
    .sum()
    .reset_index()
    .groupby('month')['PRECTOTCORR']
    .mean()
    .reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(gs_monthly_precip['month'], gs_monthly_precip['PRECTOTCORR'],
       color='steelblue', edgecolor='white')
ax.set_xlabel('Month')
ax.set_ylabel('Total Precipitation (mm)')
ax.set_title(f'Monthly Precipitation — {latest_year} Growing Season (mean across fields)')
ax.set_xticks(GROWING_MONTHS)
ax.set_xticklabels(['May', 'Jun', 'Jul', 'Aug', 'Sep'])
plt.tight_layout()
plt.savefig('output/06_monthly_precipitation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_monthly_precipitation.png')

In [ ]:
# Growing-season precipitation total per field (most recent year)
gs_precip_by_field = (weather_gs
    .groupby('field_id')['PRECTOTCORR']
    .sum()
    .reset_index()
    .rename(columns={'PRECTOTCORR': 'gs_precip_mm'}))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(gs_precip_by_field['field_id'].astype(str),
        gs_precip_by_field['gs_precip_mm'], color='steelblue')
ax.set_xlabel('Total Precipitation (mm)')
ax.set_ylabel('Field ID')
ax.set_title(f'{latest_year} Growing-Season Precipitation by Field')
plt.tight_layout()
plt.savefig('output/06_growing_season_precip_by_field.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_growing_season_precip_by_field.png')

## 6. Solar radiation and humidity

In [ ]:
# Monthly mean solar radiation and humidity for the most recent growing season
gs_monthly_solar = weather_gs.groupby('month')['ALLSKY_SFC_SW_DWN'].mean().reset_index()
gs_monthly_rh = weather_gs.groupby('month')['RH2M'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Solar radiation
axes[0].plot(gs_monthly_solar['month'], gs_monthly_solar['ALLSKY_SFC_SW_DWN'],
             marker='o', color='orange', linewidth=2)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Solar Radiation (MJ/m²/day)')
axes[0].set_title(f'Monthly Mean Solar Radiation ({latest_year} Growing Season)')
axes[0].set_xticks(GROWING_MONTHS)
axes[0].set_xticklabels(['May', 'Jun', 'Jul', 'Aug', 'Sep'])

# Relative humidity
axes[1].plot(gs_monthly_rh['month'], gs_monthly_rh['RH2M'],
             marker='o', color='teal', linewidth=2)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Relative Humidity (%)')
axes[1].set_title(f'Monthly Mean Relative Humidity ({latest_year} Growing Season)')
axes[1].set_xticks(GROWING_MONTHS)
axes[1].set_xticklabels(['May', 'Jun', 'Jul', 'Aug', 'Sep'])

plt.tight_layout()
plt.savefig('output/06_solar_humidity_monthly.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_solar_humidity_monthly.png')

## 7. Growing Degree Days (GDD)

In [ ]:
def calculate_gdd(df, base_temp=10.0, cap_temp=30.0):
    """Calculate daily and cumulative GDD per field per year.

    Formula: GDD = max(0, min((T2M_MIN + T2M_MAX) / 2, cap_temp) - base_temp)
    """
    out = df.copy()
    t_avg = ((out['T2M_MIN'] + out['T2M_MAX']) / 2).clip(upper=cap_temp)
    out['gdd'] = (t_avg - base_temp).clip(lower=0)
    out = out.sort_values(['field_id', 'date'])
    out['gdd_cumulative'] = out.groupby('field_id')['gdd'].cumsum()
    return out


# GDD for the most recent growing season (base=10°C, cap=30°C)
weather_gs_gdd = calculate_gdd(weather_gs, base_temp=10.0, cap_temp=30.0)
print(f'GDD calculated for {weather_gs_gdd["field_id"].nunique()} fields '
      f'over {latest_year} growing season.')
weather_gs_gdd[['field_id', 'date', 'T2M_MIN', 'T2M_MAX', 'gdd', 'gdd_cumulative']].head(10)

In [ ]:
# Total growing-season GDD per field
gs_total_gdd = (weather_gs_gdd
    .groupby('field_id')['gdd']
    .sum()
    .reset_index()
    .rename(columns={'gdd': 'total_gdd'}))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(gs_total_gdd['field_id'].astype(str), gs_total_gdd['total_gdd'],
        color='coral', edgecolor='white')
ax.set_xlabel('Total GDD (base 10°C)')
ax.set_ylabel('Field ID')
ax.set_title(f'{latest_year} Growing-Season GDD by Field')
plt.tight_layout()
plt.savefig('output/06_growing_season_gdd_by_field.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_growing_season_gdd_by_field.png')

In [ ]:
# Cumulative GDD curves for all fields — most recent growing season
fig, ax = plt.subplots(figsize=(14, 6))
for fid in weather_gs_gdd['field_id'].unique():
    fdata = weather_gs_gdd[weather_gs_gdd['field_id'] == fid]
    ax.plot(fdata['date'], fdata['gdd_cumulative'], alpha=0.6, linewidth=1.0)

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative GDD (base 10°C)')
ax.set_title(f'Cumulative GDD — All Fields ({latest_year} Growing Season, May–Sep)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig('output/06_cumulative_gdd_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_cumulative_gdd_curves.png')

## 8. Growing-season weather summary

In [ ]:
# Growing-season summary per field for the most recent year
growing_summary = (weather_gs_gdd
    .groupby('field_id')
    .agg(
        mean_temp=('T2M', 'mean'),
        max_temp=('T2M_MAX', 'max'),
        total_precip=('PRECTOTCORR', 'sum'),
        mean_solar=('ALLSKY_SFC_SW_DWN', 'mean'),
        mean_rh=('RH2M', 'mean'),
        total_gdd=('gdd', 'sum'),
    )
    .reset_index())

growing_summary['year'] = latest_year

print(f'Growing-season summary: {growing_summary.shape[0]} fields for {latest_year}')
growing_summary.describe()

In [ ]:
# Growing-season weather heatmap (field × variable)
heat_cols = ['mean_temp', 'max_temp', 'total_precip', 'mean_solar', 'mean_rh', 'total_gdd']
heat_data = growing_summary.set_index('field_id')[heat_cols]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heat_data, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax)
ax.set_title(f'{latest_year} Growing-Season Weather Summary by Field')
ax.set_ylabel('Field ID')
plt.tight_layout()
plt.savefig('output/06_growing_season_summary_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_growing_season_summary_heatmap.png')

## 9. Weather by crop type

In [ ]:
# Merge growing-season summary with crop type for the most recent year
cdl_latest = cdl[cdl['year'] == latest_year][['field_id', 'year', 'crop_name']]
growing_crop = growing_summary.merge(
    cdl_latest,
    on=['field_id', 'year'],
    how='left'
)
print(f'Merged records: {growing_crop.shape[0]}')
print(f'Crop types: {growing_crop["crop_name"].value_counts().to_dict()}')

In [ ]:
# Compare growing-season GDD and precipitation by crop type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=growing_crop, x='crop_name', y='total_gdd', ax=axes[0])
axes[0].set_xlabel('Crop Type')
axes[0].set_ylabel('Growing-Season GDD (base 10°C)')
axes[0].set_title(f'GDD by Crop Type ({latest_year})')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=growing_crop, x='crop_name', y='total_precip', ax=axes[1])
axes[1].set_xlabel('Crop Type')
axes[1].set_ylabel('Growing-Season Precipitation (mm)')
axes[1].set_title(f'Precipitation by Crop Type ({latest_year})')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('output/06_weather_by_crop_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_weather_by_crop_type.png')

## 10. Weather variable correlations

In [ ]:
# Correlation matrix of growing-season weather variables
corr_cols = ['mean_temp', 'max_temp', 'total_precip', 'mean_solar', 'mean_rh', 'total_gdd']
corr_matrix = growing_summary[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title(f'{latest_year} Growing-Season Weather Correlations')
plt.tight_layout()
plt.savefig('output/06_weather_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_weather_correlation_matrix.png')

In [ ]:
# Scatter: growing-season GDD vs precipitation, colored by crop type
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=growing_crop, x='total_gdd', y='total_precip',
                hue='crop_name', s=100, ax=ax)
ax.set_xlabel('Growing-Season GDD (base 10°C)')
ax.set_ylabel('Growing-Season Precipitation (mm)')
ax.set_title(f'GDD vs Precipitation by Crop Type ({latest_year})')
ax.legend(title='Crop')
plt.tight_layout()
plt.savefig('output/06_gdd_vs_precip_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved output/06_gdd_vs_precip_scatter.png')

## 11. Summary and key findings

In [ ]:
# Summary statistics for the most recent growing season
print(f'=== {latest_year} Growing-Season Weather Summary (May–Sep) ===')
print(f'Fields analyzed: {growing_summary["field_id"].nunique()}')
print(f'Date range: {weather_gs["date"].min().date()} to {weather_gs["date"].max().date()}')
print()
print(f'Mean temperature:    {growing_summary["mean_temp"].mean():.1f}°C')
print(f'Max temperature:     {growing_summary["max_temp"].mean():.1f}°C')
print(f'Total GDD:           {growing_summary["total_gdd"].mean():.0f} (mean across fields)')
print(f'Total precipitation: {growing_summary["total_precip"].mean():.0f} mm (mean across fields)')
print(f'Mean solar radiation:{growing_summary["mean_solar"].mean():.1f} MJ/m²/day')
print(f'Mean rel. humidity:  {growing_summary["mean_rh"].mean():.1f}%')

## 12. Export processed data

In [ ]:
# Save growing-season summary for downstream use
growing_summary.to_csv('output/06_growing_season_summary.csv', index=False)
print(f'Exported: output/06_growing_season_summary.csv ({growing_summary.shape[0]} rows)')

# Save filtered growing-season weather with GDD columns
weather_gs_gdd.to_csv('output/06_weather_growing_season_with_gdd.csv', index=False)
print(f'Exported: output/06_weather_growing_season_with_gdd.csv ({weather_gs_gdd.shape[0]:,} rows)')